In [3]:
import os
import glob
import polars as pl
import numpy as np
from pathlib import Path

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
# Raw data directory (needed to extract ground-truth 'round_end' winners)
RAW_DATA_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\datasets\billpureskillgg\cs2-2023-11-23\versions\2\csds")

# Feature Data Directories (Inputs)
TRAIN_FEAT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Training\Dust2")
#TEST_FEAT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\FeatData\Wfire_STOMS_01\Testing\Dust2")

# Results Directories (Outputs)
TRAIN_OUT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\Results\STOMs\WP_DAC_02\Training\Dust2")
#TEST_OUT_DIR = Path(r"D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\Results\STOMs\WP_DAC_02\Testing\Dust2")

EPOCHS = [0, 10, 20, 30, 40, 50, 60, "End"]
GRID_SIZE = 10

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# ==========================================
# 2. GROUND TRUTH RETRIEVAL
# ==========================================
def get_actual_winners(match_id):
    """Finds the raw round_end parquet file to extract ground truth results for the match."""
    match_dirs = list(RAW_DATA_DIR.rglob(match_id))
    if not match_dirs:
        return {}

    match_dir = match_dirs[0]
    re_files = glob.glob(str(match_dir / "round_end*"))
    
    if not re_files:
        return {}

    # Load and standardize winner team codes
    try:
        df_end = pl.read_parquet(re_files[0])
        team_map = {"2": "T", "3": "CT", "TERRORIST": "T", "CT": "CT", "T": "T"}
        
        df_end = df_end.with_columns(
            pl.col("winner_team_code").cast(pl.Utf8).replace(team_map).alias("actual_winner")
        ).select(["round", "actual_winner"])
        
        return dict(zip(df_end["round"].to_list(), df_end["actual_winner"].to_list()))
    except Exception as e:
        print(f"Error reading raw labels for {match_id}: {e}")
        return {}

# ==========================================
# 3. CORE PROCESSING LOGIC
# ==========================================
def process_feature_directory(input_dir, output_dir):
    """Processes all match CSVs, constructs the outcome matrix per round, and saves them."""
    
    # Ensure output directory exists
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Get all match feature files
    match_files = list(input_dir.glob("*.csv"))
    
    if not match_files:
        print(f"No CSV files found in: {input_dir}")
        return

    print(f"\nProcessing {len(match_files)} matches from {input_dir.parent.name}/{input_dir.name}...")

    for filepath in match_files:
        match_id = filepath.stem
        
        try:
            df_feat = pl.read_csv(filepath)
            actual_winners_map = get_actual_winners(match_id)
            
            rounds = df_feat["round"].to_list()
            
            # Dictionary to build the final DataFrame (Outcome Matrix)
            # Column 1: Round ID
            outcome_data = {"round": rounds}
            
            # Initialize lists for aggregated cells, probabilities, and predictions vs epochs
            for team in ["CT", "T"]:
                for e in EPOCHS:
                    outcome_data[f"{team}_agg_cells_{e}"] = []
                    
            for e in EPOCHS:
                outcome_data[f"Prob_CT_{e}"] = []
                outcome_data[f"Pred_{e}"] = []
                
            actual_winner_list = []

            # Iterate through every round (row) in the match
            for r_idx in range(len(rounds)):
                round_data = df_feat.row(r_idx, named=True)
                r_id = round_data["round"]
                actual_winner = actual_winners_map.get(r_id, "Unknown")
                actual_winner_list.append(actual_winner)

                for epoch in EPOCHS:
                    # Spatially collapsed counts (Sum of grids with activity > 0)
                    ct_occ = sum(1 for i in range(GRID_SIZE**2) if round_data.get(f"CT_e{epoch}_cell_{i:02d}", 0) > 0)
                    t_occ = sum(1 for i in range(GRID_SIZE**2) if round_data.get(f"T_e{epoch}_cell_{i:02d}", 0) > 0)
                    
                    # Compute CT win probability using sigmoid of the difference
                    prob_ct = sigmoid(ct_occ - t_occ)
                    
                    # Make Prediction based on > 0.5 threshold
                    pred_winner = "CT" if prob_ct > 0.5 else "T"
                    
                    # Append to tracking lists
                    outcome_data[f"CT_agg_cells_{epoch}"].append(ct_occ)
                    outcome_data[f"T_agg_cells_{epoch}"].append(t_occ)
                    outcome_data[f"Prob_CT_{epoch}"].append(prob_ct)
                    outcome_data[f"Pred_{epoch}"].append(pred_winner)

            # Final Column: Actual Winner
            outcome_data["actual_winner"] = actual_winner_list
            
            # Construct DataFrame and enforce exact column order
            df_outcome = pl.DataFrame(outcome_data)
            
            ordered_cols = ["round"]
            ordered_cols += [f"CT_agg_cells_{e}" for e in EPOCHS]
            ordered_cols += [f"T_agg_cells_{e}" for e in EPOCHS]
            ordered_cols += [f"Prob_CT_{e}" for e in EPOCHS]
            ordered_cols += [f"Pred_{e}" for e in EPOCHS]
            ordered_cols += ["actual_winner"]
            
            df_outcome = df_outcome.select(ordered_cols)
            
            # Save the match specific results
            out_filepath = output_dir / f"{match_id}.csv"
            df_outcome.write_csv(out_filepath)
            
        except Exception as e:
            print(f"Error processing file {match_id}: {e}")
            
    print(f"Finished saving matrices to: {output_dir}")

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    # Step 1-5: Process Training Data
    process_feature_directory(TRAIN_FEAT_DIR, TRAIN_OUT_DIR)
    
    # Step 6: Repeat for Testing Data
    #process_feature_directory(TEST_FEAT_DIR, TEST_OUT_DIR)
    
    print("\nPipeline execution complete.")


Processing 87 matches from Training/Dust2...
Finished saving matrices to: D:\Research_Data_Public\Esports\CSDATA\DataScience\CS2_Project\Results\STOMs\WP_DAC_02\Training\Dust2

Pipeline execution complete.
